# RAG Implementation Chatbot - Using OpenAI API
## Get Information about Scholarships provided by Govt. of India
### https://scholarships.gov.in/


1.  **Dataset Loading**: The project utilizes a dataset of Indian government scholarships, sourced from scholarships.gov.in and made available on Hugging Face (`NetraVerse/indian-govt-scholarships`).

2.  **Data Chunking**: The raw text data from the scholarships is split into smaller, manageable chunks to improve the relevance and efficiency of information retrieval.

3.  **Vector Database**: [Qdrant](https://qdrant.tech/) is employed as the vector database to store and manage the embeddings of these document chunks.

4.  **Embedding Generation**: [SentenceTransformer](https://www.sbert.net/) (`all-MiniLM-L6-v2`) is used to convert both the scholarship document chunks and user queries into dense vector representations (embeddings). This model maps sentences and paragraphs into a 384-dimensional vector space, enabling semantic similarity search.

5.  **Retrieval**: When a user poses a query, its embedding is generated and used to search the Qdrant vector database. The system retrieves the most semantically similar document chunks to the query.

6.  **Language Model (LLM)**: OpenAI ChatGPT 3.5 Turbo is the chosen Large Language Model. It's a smaller, efficient model capable of generating coherent responses.

7.  **Response Generation (RAG)**: The retrieved document chunks are provided as context to TinyLlama. The LLM then generates a factual and relevant answer to the user's query, grounded in the provided scholarship information.

8.  **Interactive Interface**: The entire RAG pipeline is encapsulated within a [Gradio](https://www.gradio.app/) interface, allowing users to interact with the chatbot in real-time.


## Step 1: Install Required Dependencies

In [ ]:
# Install required packages
!pip install pandas qdrant-client sentence-transformers openai gradio

## Step 2: Set Up OpenAI API Key

In [ ]:
import os
from google.colab import userdata

# Get your API key from Colab Secrets (recommended for security)
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✓ OpenAI API key configured")

## Step 3: Load the Dataset

In [ ]:
import pandas as pd
from pprint import pprint
from datasets import load_dataset

# Read scholarship data from Hugging Face
print("Loading scholarship dataset...")
dataset = load_dataset("NetraVerse/indian-govt-scholarships")
df = dataset["train"].to_pandas()
df = df[['label', 'text']]

# Convert to dict format
data = df.to_dict('records')
print(f"✓ Loaded {len(data)} scholarship documents")
print(f"\nFirst document example:")
print(f"Label: {data[0]['label']}")
print(f"Text (first 200 chars): {data[0]['text'][:200]}...")

## Step 4: Chunk the Data

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    '''Split text into overlapping chunks'''
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunked_data = []
for doc in data:
    text = doc['text']
    chunks = chunk_text(text, chunk_size=500, overlap=100)
    for i, chunk in enumerate(chunks):
         chunked_data.append({
         'label': doc['label'],
         'text': chunk,
         'chunk_id': i,
         'total_chunks': len(chunks)
     })
data = chunked_data
print(f"✓ Chunked into {len(data)} pieces")


## Step 5: Initialize Vector Database (Qdrant) and Embeddings

In [ ]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Initialize Qdrant (in-memory)
qdrant = QdrantClient(":memory:")

# Initialize embedding model
# Note: Using local SentenceTransformer for embeddings (lightweight, no API needed)
print("Loading embedding model...")
encoder_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Embedding model loaded")
print(f"✓ Vector dimension: {encoder_model.get_sentence_embedding_dimension()}")

# Create Qdrant collection
collection_name = "scholarships"
qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder_model.get_sentence_embedding_dimension(),
        distance=models.Distance.COSINE
    )
)
print(f"✓ Qdrant collection '{collection_name}' created")

## Step 6: Generate Embeddings and Upload to Vector Database

In [ ]:
print("Generating embeddings for all documents...")
points_to_upload = []
for idx, doc in enumerate(data):
    points_to_upload.append(
        models.PointStruct(
            id=idx,
            vector=encoder_model.encode(doc["text"]).tolist(),
            payload=doc
        )
    )
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(data)} documents...")

# Upload to Qdrant
qdrant.upload_points(
    collection_name=collection_name,
    points=points_to_upload
)

count = qdrant.count(collection_name=collection_name, exact=True).count
print(f"✓ Uploaded {count} embeddings to Qdrant")

## Step 7: Initialize OpenAI API Client

In [ ]:
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("✓ OpenAI API client initialized")

def generate_response(user_query, search_results):
    """
    Generate response using OpenAI API
    Args:
        user_query: The user's question
        search_results: Retrieved document chunks from vector database
    Returns:
        Response text from OpenAI
    """
    system_prompt = f"""You are a helpful chatbot specializing in Indian government scholarships.
Use the following retrieved documents to answer the user's question accurately.
ONLY use information from the retrieved documents. If the documents don't contain the answer, say so.

Retrieved Documents:
{str(search_results)}
"""
    #system_prompt = f"you are helpful assistant"
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",  # Change to "gpt-4" for better quality (more expensive)
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating response: {str(e)}"

print("✓ Response generation function ready")

## Step 8: Create RAG Chatbot Function

In [ ]:
def scholarship_chatbot(message, history):
    """
    Main RAG chatbot function
    Args:
        message: User's input message
        history: Chat history (provided by Gradio)
    Returns:
        Response with sources
    """
    # Step 1: Encode user query to vector
    query_vector = encoder_model.encode(message).tolist()

    # Step 2: Search vector database for relevant chunks
    hits = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=3  # Retrieve top 3 most relevant documents
    )

    # Step 3: Extract search results and collect sources
    search_results = []
    sources_list = []
    for hit in hits.points:
        search_results.append(hit.payload)
        if hit.payload['label'] not in sources_list:
            sources_list.append(hit.payload['label'])

    # Step 4: Generate response using OpenAI API
    response = generate_response(message, search_results)

    # Step 5: Format response with sources
    sources = " | ".join(sources_list) if sources_list else "No sources found"
    response_with_sources = f"{response}\n\n📚 **Sources:** {sources}"

    return response_with_sources

print("✓ RAG chatbot function created")

## Step 9: Test the Chatbot (Optional)

In [ ]:
# Test with a sample query
test_query = "What scholarships are available for engineering students?"
print(f"Testing with query: {test_query}\n")
response = scholarship_chatbot(test_query, [])
print("Response:")
print(response)

## Step 10: Launch Gradio Interface

In [ ]:
import gradio as gr

# Create Gradio interface
demo = gr.ChatInterface(
    scholarship_chatbot,
    title="📚 Indian Government Scholarship Chatbot",
    description="Ask me about Indian government scholarships! This chatbot uses RAG (Retrieval-Augmented Generation) with OpenAI API to provide accurate information from official scholarship data.",
    examples=[
        "What scholarships are available for engineering students?",
        "Tell me about AICTE scholarships",
        "Are there scholarships for women in STEM?",
        "What is the percentage of reservations in NSPG Scheme?",
        "How do I apply for government scholarships?",
        "What is the leave policy in the indian government"
    ],
    theme=gr.themes.Soft()
)

# Launch the interface
demo.launch(share=False)  # Set share=True to get a public link

## 📋 Configuration & Customization

### Change OpenAI Model
In Step 7, change the model parameter:
```python
model="gpt-4",  # Higher quality but more expensive
model="gpt-3.5-turbo",  # Faster and cheaper
model="gpt-4-turbo",  # Balance of quality and cost
```

### Adjust Response Quality
In Step 7, modify these parameters:
- `max_tokens`: Default 500. Increase for longer responses
- `temperature`: Default 0.7. Range 0-1. Lower = more focused, Higher = more creative
- `limit`: In Step 8, increase from 3 to 5-10 for more context

### API Pricing
- GPT-3.5-turbo: ~$0.0005 per 1K tokens (very cheap)
- GPT-4: ~$0.03 per 1K tokens (more expensive)
- Check current pricing: https://openai.com/pricing

### Troubleshooting
1. **Invalid API Key**: Check https://platform.openai.com/api-keys
2. **Rate Limits**: OpenAI has rate limits. Add delay: `import time; time.sleep(1)`
3. **Token Limits**: If responses are cut off, increase `max_tokens`
4. **Memory Issues**: Reduce chunk size or number of documents
